In [ ]:
# 依赖导入。为了支持 Run All，这里不自动 pip install；缺包时只提示，相关章节会自动跳过。
import importlib

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from zoneinfo import ZoneInfo

OPTIONAL_DEPENDENCIES = {}
for package_name in ["vnpy", "baostock", "akshare"]:
    try:
        OPTIONAL_DEPENDENCIES[package_name] = importlib.import_module(package_name)
        print(f"{package_name} available")
    except Exception as e:
        OPTIONAL_DEPENDENCIES[package_name] = None
        print(f"{package_name} unavailable:", repr(e))

vnpy = OPTIONAL_DEPENDENCIES["vnpy"]
bs = OPTIONAL_DEPENDENCIES["baostock"]
ak = OPTIONAL_DEPENDENCIES["akshare"]

try:
    from vnpy.trader.object import BarData
    from vnpy.trader.constant import Exchange, Interval
    from vnpy.trader.database import get_database
    VNPY_IMPORT_AVAILABLE = True
except Exception as e:
    VNPY_IMPORT_AVAILABLE = False
    print("vn.py trader imports unavailable:", repr(e))


In [ ]:
df = pd.read_parquet("688981_5min_20200716-20260602.parquet")
df.head(100)

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
float_cols = ["open", "high", "low", "close", "volume", "amount"]


# 这个errors的意思是遇到不能转换的格式，就变成NaN，而不是报错。
df[float_cols] = df[float_cols].apply(pd.to_numeric, errors="coerce")

In [ ]:
df.info()

In [ ]:
# =========================
# 特征工程：模块化构造
# =========================

from feature_engineering import build_basic_features

df = build_basic_features(df)
print("特征工程后数据量:", df.shape)
display(df.head())


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 8 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 9 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 10 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 11 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 12 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 13 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 14 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 15 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 16 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 17 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 18 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 19 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 20 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 21 保留为空，避免 Run All 顺序变化。


In [ ]:
# 已迁移到 feature_engineering.build_basic_features
# 原 cell 22 保留为空，避免 Run All 顺序变化。


In [ ]:
# =========================
# MVP 全局参数与路径依赖机会标签构造
# =========================

import importlib
import mvp_config
importlib.reload(mvp_config)
from mvp_config import build_mvp_config
from label_builder import build_path_dependent_opportunity_labels, summarize_opportunity_labels

# 后续调参优先改 mvp_config.py；临时试验可在 overrides 里覆盖。
MVP_CONFIG = build_mvp_config()

HORIZON = MVP_CONFIG["horizon"]
COMMISSION_RATE = MVP_CONFIG["commission_rate"]
SLIPPAGE_RATE = MVP_CONFIG["slippage_rate"]
ONE_SIDE_COST = MVP_CONFIG["one_side_cost"]
ROUND_TRIP_COST = MVP_CONFIG["round_trip_cost"]
LABEL_COST = MVP_CONFIG["label_cost"]
LABEL_THRESHOLD = MVP_CONFIG["label_threshold"]
TOTAL_COST = MVP_CONFIG["total_cost"]

BUY_CLASS = MVP_CONFIG["buy_class"]
SELL_POSITIVE_CLASS = 1  # positive class for standalone sell_label
SELL_CLASS = MVP_CONFIG.get("sell_class", None)  # only used by deprecated multiclass action labels
BUY_TARGET_COL = MVP_CONFIG["buy_label_col"]
SELL_TARGET_COL = MVP_CONFIG["sell_label_col"]

# 构造 triple-barrier 路径依赖机会标签。
df = build_path_dependent_opportunity_labels(
    df,
    horizon=HORIZON,
    tp=MVP_CONFIG["tp"],
    sl=MVP_CONFIG["sl"],
    cost=LABEL_COST,
    same_bar_policy=MVP_CONFIG["same_bar_policy"],
    timeout_policy=MVP_CONFIG["timeout_policy"],
    buy_label_col=BUY_TARGET_COL,
    sell_label_col=SELL_TARGET_COL,
)

# 兼容旧绘图/诊断命名：trade_return 仍表示固定 HORIZON close 的买入收益，仅作参考，不作为主标签。
df["future_log_return"] = np.log(df["future_close"] / df["close"])
df["next_ret_1"] = df["close"].shift(-1) / df["close"] - 1

if MVP_CONFIG.get("drop_incomplete_labels", True):
    before_rows = len(df)
    df = df.dropna(subset=["future_entry_open", "future_close"]).copy()
    print("删除未来路径不完整样本:", before_rows - len(df))

label_summary_current, label_reason_distribution = summarize_opportunity_labels(
    df,
    buy_label_col=BUY_TARGET_COL,
    sell_label_col=SELL_TARGET_COL,
)

print("MVP_CONFIG:")
print(pd.Series(MVP_CONFIG))
print("\n单边成本 ONE_SIDE_COST / LABEL_COST:", LABEL_COST)
print("往返成本 ROUND_TRIP_COST:", ROUND_TRIP_COST)
print("\n路径机会标签分布:")
display(label_summary_current)
print("\n触发原因分布:")
display(label_reason_distribution)
print("\nbuy_label value_counts:")
print(df[BUY_TARGET_COL].value_counts(normalize=True).sort_index())
print("\nsell_label value_counts:")
print(df[SELL_TARGET_COL].value_counts(normalize=True).sort_index())


In [ ]:
# =========================
# 特征选择：模块化筛选
# =========================

from feature_engineering import select_feature_columns

feature_cols, feature_selection_report = select_feature_columns(df)
print("删除绝对水平类特征数量:", len(feature_selection_report["absolute_level_cols"]))
print(feature_selection_report["absolute_level_cols"])
print("疑似未来函数字段:", feature_selection_report["leak_cols"])
print("常数特征数量:", len(feature_selection_report["constant_cols"]))
print(feature_selection_report["constant_cols"])
print("最终特征数量:", len(feature_cols))
print(feature_cols)


In [ ]:
df.head()

In [ ]:
# 已迁移到 feature_engineering.select_feature_columns
# 原 cell 26 保留为空，避免重复筛选 feature_cols。


In [ ]:
# 已迁移到 feature_engineering.select_feature_columns
# 原 cell 27 保留为空，避免重复筛选 feature_cols。


In [ ]:
# 已迁移到 feature_engineering.select_feature_columns
# 原 cell 28 保留为空，避免重复筛选 feature_cols。


In [ ]:
# 已迁移到 feature_engineering.select_feature_columns
# 原 cell 29 保留为空，避免重复筛选 feature_cols。


In [ ]:
# =========================
# 时间序列切分
# =========================

from feature_engineering import split_time_series

target_col = MVP_CONFIG["target_col"]
sell_target_col = MVP_CONFIG["sell_target_col"]
assert target_col in df.columns, f"target_col 不存在: {target_col}"
assert sell_target_col in df.columns, f"sell_target_col 不存在: {sell_target_col}"

df_model = df.copy().sort_index()
train_df, valid_df, test_df = split_time_series(
    df_model,
    train_ratio=MVP_CONFIG["train_ratio"],
    valid_ratio=MVP_CONFIG["valid_ratio"],
)

print("当前 target_col:", target_col)
print("sell_target_col:", sell_target_col)
print("标签定义: path-dependent triple-barrier opportunity labels")
print("HORIZON:", HORIZON, "TP:", MVP_CONFIG["tp"], "SL:", MVP_CONFIG["sl"], "LABEL_COST:", LABEL_COST)
print("训练集：", train_df.index.min(), train_df.index.max(), train_df.shape)
print("验证集：", valid_df.index.min(), valid_df.index.max(), valid_df.shape)
print("测试集：", test_df.index.min(), test_df.index.max(), test_df.shape)

for split_name, part in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    print(f"\n{split_name} buy_label 分布:")
    print(part[target_col].value_counts(normalize=True).sort_index())
    print(f"{split_name} sell_label 分布:")
    print(part[sell_target_col].value_counts(normalize=True).sort_index())


In [ ]:
# =========================
# Label 诊断并导出
# =========================

from pathlib import Path
from diagnostics import export_label_diagnostics

LABEL_DIAG_DIR = Path(MVP_CONFIG["diagnostics_dir"])
split_frames = {"all": df_model, "train": train_df, "valid": valid_df, "test": test_df}

(
    label_summary,
    label_reason_distribution,
    label_return_by_class,
    label_parameter_stability,
    label_diag_summary,
) = export_label_diagnostics(
    split_frames=split_frames,
    buy_label_col=BUY_TARGET_COL,
    sell_label_col=SELL_TARGET_COL,
    config=MVP_CONFIG,
    output_dir=LABEL_DIAG_DIR,
)

print("Label 诊断结果已保存到:", LABEL_DIAG_DIR)
display(label_summary)
display(label_reason_distribution)
display(label_return_by_class)
display(label_parameter_stability.head(20))


## MVP Current Experiment Setup

Current focus: finish the machine-learning signal layer before optimizing the trading/backtest rules.

```text
LOOKBACK = 32
HORIZON = 3
target_col = buy_label
sell_target_col = sell_label
label definition = path-dependent triple-barrier opportunity labels
sequence_mode = continuous
```

Interpretation:

- `buy_label = 1`: from the next bar open, the future path contains a tradable buy/low-entry opportunity.
- `sell_label = 1`: from the next bar open, the future path contains a tradable sell/high-exit opportunity.
- The current main LSTM/RF/CNN/MLP path still trains `buy_label`; the signal layer should next produce both `buy_prob` and `sell_prob`.


In [ ]:
# =========================
# 当前 MVP 实验配置表
# =========================

experiment_grid = pd.DataFrame([
    {
        "lookback": MVP_CONFIG["lookback"],
        "horizon": MVP_CONFIG["horizon"],
        "target_col": MVP_CONFIG["target_col"],
        "sell_target_col": MVP_CONFIG["sell_target_col"],
        "label_definition": "path-dependent triple-barrier opportunity labels: buy_label/sell_label",
        "tp": MVP_CONFIG["tp"],
        "sl": MVP_CONFIG["sl"],
        "label_cost": LABEL_COST,
        "round_trip_cost": ROUND_TRIP_COST,
        "same_bar_policy": MVP_CONFIG["same_bar_policy"],
        "timeout_policy": MVP_CONFIG["timeout_policy"],
        "fixed_threshold_quantile": MVP_CONFIG["fixed_threshold_quantile"],
        "sequence_mode": MVP_CONFIG["sequence_mode"],
        "status": "active",
    }
])

experiment_grid["lookback_minutes"] = experiment_grid["lookback"] * MVP_CONFIG["bar_minutes"]
experiment_grid["horizon_minutes"] = experiment_grid["horizon"] * MVP_CONFIG["bar_minutes"]

Path("outputs/diagnostics").mkdir(parents=True, exist_ok=True)
experiment_grid.to_csv("outputs/diagnostics/experiment_grid_plan.csv", index=False)

print("当前 MVP 实验配置:")
display(experiment_grid)


In [ ]:
# =========================
# 用训练集分位数缩尾
# =========================

from feature_engineering import winsorize_by_train

train_df, valid_df, test_df, clip_bounds = winsorize_by_train(
    train_df,
    valid_df,
    test_df,
    feature_cols,
    lower_q=MVP_CONFIG["winsor_lower_q"],
    upper_q=MVP_CONFIG["winsor_upper_q"],
)
print("缩尾完成")
print("缩尾特征数量:", len(clip_bounds))


In [ ]:
# =========================
# 特征量纲检测：标准化前
# =========================

from diagnostics import feature_scale_report

pre_scale_report = feature_scale_report(train_df, feature_cols)
print("特征数量:", len(feature_cols))
print("缺失率最高的特征:")
display(pre_scale_report.sort_values("missing_rate", ascending=False).head(15))
print("abs_max 最大的特征，也就是量纲/极值最大的特征:")
display(pre_scale_report.sort_values("abs_max", ascending=False).head(20))
print("std 最大的特征:")
display(pre_scale_report.sort_values("std", ascending=False).head(20))
print("近似常数或波动极小的特征:")
display(pre_scale_report.sort_values("std", ascending=True).head(20))

scale_warn_cols = pre_scale_report.index[
    (pre_scale_report["abs_max"] > 1e6) |
    (pre_scale_report["std"] > 1e4) |
    (pre_scale_report["missing_rate"] > 0.05) |
    (pre_scale_report["nunique"] <= 2)
].tolist()
print("可能需要关注的特征数量:", len(scale_warn_cols))
print(scale_warn_cols[:80])


In [ ]:
# =========================
# 特征标准化
# =========================

from feature_engineering import standardize_by_train

X_train = train_df[feature_cols]
X_valid = valid_df[feature_cols]
X_test = test_df[feature_cols]
y_train = train_df[target_col]
y_valid = valid_df[target_col]
y_test = test_df[target_col]

train_scaled, valid_scaled, test_scaled, scaler = standardize_by_train(
    train_df,
    valid_df,
    test_df,
    feature_cols,
)

print("标准化完成")
print("特征数量:", len(feature_cols))


In [ ]:
# =========================
# 特征最终检查
# =========================

print("是否存在缺失值：", train_scaled[feature_cols].isna().any().any())
print("是否存在无穷值：", np.isinf(train_scaled[feature_cols]).any().any())
print("训练集特征均值最大值：")
print(train_scaled[feature_cols].mean().abs().sort_values(ascending=False).head(10))
print("训练集特征标准差：")
print(train_scaled[feature_cols].std().sort_values(ascending=False).head(10))
print("最终特征数量:", len(feature_cols))


In [ ]:
# =========================
# 特征量纲检测：标准化后
# =========================

from diagnostics import scaled_feature_report

post_scale_report = scaled_feature_report(train_scaled, feature_cols)
print("标准化后均值绝对值最大的特征:")
display(post_scale_report.sort_values("mean_abs", ascending=False).head(20))
print("标准化后std最偏离1的特征:")
display(post_scale_report.sort_values("std_dev_from_1", ascending=False).head(20))
print("标准化后abs_max最大的特征，可能是极端异常点残留:")
display(post_scale_report.sort_values("abs_max", ascending=False).head(20))

bad_scaled_cols = post_scale_report.index[
    (post_scale_report["mean_abs"] > 1e-6) |
    (post_scale_report["std_dev_from_1"] > 1e-3) |
    (post_scale_report["missing_rate"] > 0) |
    (post_scale_report["abs_max"] > 10)
].tolist()
print("标准化后需要关注的特征数量:", len(bad_scaled_cols))
print(bad_scaled_cols[:80])


In [ ]:
df.head()

In [ ]:
# =========================
# Sequence data aligned for buy / sell labels
# =========================

from feature_engineering import make_sequence_data, save_run_config

LOOKBACK = MVP_CONFIG["lookback"]
SEQUENCE_MODE = MVP_CONFIG["sequence_mode"]

X_train_seq, y_train_seq, train_index_seq = make_sequence_data(train_scaled, feature_cols, target_col, LOOKBACK)
X_valid_seq, y_valid_seq, valid_index_seq = make_sequence_data(valid_scaled, feature_cols, target_col, LOOKBACK)
X_test_seq, y_test_seq, test_index_seq = make_sequence_data(test_scaled, feature_cols, target_col, LOOKBACK)

_, y_train_sell_seq, train_sell_index_seq = make_sequence_data(train_scaled, feature_cols, sell_target_col, LOOKBACK)
_, y_valid_sell_seq, valid_sell_index_seq = make_sequence_data(valid_scaled, feature_cols, sell_target_col, LOOKBACK)
_, y_test_sell_seq, test_sell_index_seq = make_sequence_data(test_scaled, feature_cols, sell_target_col, LOOKBACK)

assert np.array_equal(train_index_seq, train_sell_index_seq)
assert np.array_equal(valid_index_seq, valid_sell_index_seq)
assert np.array_equal(test_index_seq, test_sell_index_seq)

print("buy target_col:", target_col)
print("sell_target_col:", sell_target_col)
print("LOOKBACK:", LOOKBACK)
print("SEQUENCE_MODE:", SEQUENCE_MODE)
print("len(feature_cols):", len(feature_cols))
print("X_train_seq:", X_train_seq.shape)
print("X_valid_seq:", X_valid_seq.shape)
print("X_test_seq:", X_test_seq.shape)
print("train buy/sell positive rate:", np.mean(y_train_seq), np.mean(y_train_sell_seq))
print("valid buy/sell positive rate:", np.mean(y_valid_seq), np.mean(y_valid_sell_seq))
print("test buy/sell positive rate:", np.mean(y_test_seq), np.mean(y_test_sell_seq))

assert X_train_seq.shape[2] == len(feature_cols), (X_train_seq.shape, len(feature_cols))
assert X_valid_seq.shape[2] == len(feature_cols), (X_valid_seq.shape, len(feature_cols))
assert X_test_seq.shape[2] == len(feature_cols), (X_test_seq.shape, len(feature_cols))

run_config = {
    "lookback": LOOKBACK,
    "horizon": HORIZON,
    "target_col": target_col,
    "sell_target_col": sell_target_col,
    "sequence_mode": SEQUENCE_MODE,
    "feature_count": len(feature_cols),
    "train_seq_count": int(len(X_train_seq)),
    "valid_seq_count": int(len(X_valid_seq)),
    "test_seq_count": int(len(X_test_seq)),
    "mvp_config": MVP_CONFIG,
}
save_run_config(run_config, MVP_CONFIG["diagnostics_dir"])


In [ ]:
# =========================
# Logistic Regression buy/sell signal baseline
# =========================

# This cell completes the first ML signal layer: buy_prob and sell_prob.
# Implementation lives in model_signals.py so the notebook stays readable.

import importlib
import json
import model_signals
importlib.reload(model_signals)
from model_signals import train_dual_logistic_signals

logit_result = train_dual_logistic_signals(
    train_scaled=train_scaled,
    valid_scaled=valid_scaled,
    test_scaled=test_scaled,
    valid_df=valid_df,
    test_df=test_df,
    feature_cols=feature_cols,
    buy_target_col=target_col,
    sell_target_col=sell_target_col,
    valid_index_seq=valid_index_seq,
    test_index_seq=test_index_seq,
    config=MVP_CONFIG,
)

buy_logit = logit_result["buy_model"]
sell_logit = logit_result["sell_model"]
logit_signal_valid = logit_result["valid_signals"]
logit_signal_test = logit_result["test_signals"]
logit_group_analysis = logit_result["group_analysis"]
logit_summary = logit_result["summary"]

print("Logistic buy/sell signals saved to:", logit_result["output_dir"])
print(json.dumps(logit_summary, ensure_ascii=False, indent=2))
display(logit_group_analysis)


In [ ]:
# =========================
# RandomForest buy/sell signal baseline
# =========================

# RF uses the same dual-signal contract as Logistic Regression:
# buy_model -> buy_prob, sell_model -> sell_prob.
# Implementation lives in model_signals.py; this cell keeps compatibility variables for later notebook cells.

import importlib
import model_signals
importlib.reload(model_signals)
from model_signals import train_dual_random_forest_signals

rf_signal_result = train_dual_random_forest_signals(
    train_scaled=train_scaled,
    valid_scaled=valid_scaled,
    test_scaled=test_scaled,
    valid_df=valid_df,
    test_df=test_df,
    feature_cols=feature_cols,
    buy_target_col=target_col,
    sell_target_col=sell_target_col,
    valid_index_seq=valid_index_seq,
    test_index_seq=test_index_seq,
    config=MVP_CONFIG,
)

rf_buy_model = rf_signal_result["buy_model"]
rf_sell_model = rf_signal_result["sell_model"]
rf_baseline_result = rf_signal_result["test_signals"]
rf_valid_signal_result = rf_signal_result["valid_signals"]
rf_group_analysis = rf_signal_result["group_analysis"]
rf_signal_threshold_diagnostics = rf_signal_result["threshold_diagnostics"]
rf_feature_importance = rf_signal_result["feature_importance"]
rf_summary = rf_signal_result["summary"]

# Compatibility variables used by downstream comparison/backtest cells.
rf_valid_prob = rf_valid_signal_result["buy_prob"].to_numpy()
rf_test_prob = rf_baseline_result["buy_prob"].to_numpy()
y_valid_rf = rf_valid_signal_result["buy_label_true"].astype(int)
y_test_rf = rf_baseline_result["buy_label_true"].astype(int)
y_valid_rf_buy = y_valid_rf
y_test_rf_buy = y_test_rf
rf_valid_auc = rf_summary["valid_buy_auc"]
rf_test_auc = rf_summary["test_buy_auc"]
rf_threshold_diagnostics = (
    rf_signal_threshold_diagnostics
    .loc[rf_signal_threshold_diagnostics["side"] == "buy"]
    .drop(columns=["side"])
    .reset_index(drop=True)
)

print("RandomForest buy/sell signals saved to:", rf_signal_result["output_dir"])
print(json.dumps(rf_summary, ensure_ascii=False, indent=2))
display(rf_group_analysis)
display(rf_threshold_diagnostics)
display(rf_feature_importance.groupby("side", group_keys=False).head(20))


In [ ]:
# =========================
# 序列输入主导性检测
# =========================

from diagnostics import sequence_scale_report

seq_scale_report = sequence_scale_report(X_train_seq, feature_cols)
print("序列输入 abs_mean 最大的特征:")
display(seq_scale_report.sort_values("seq_abs_mean", ascending=False).head(20))
print("序列输入 abs_max 最大的特征:")
display(seq_scale_report.sort_values("seq_abs_max", ascending=False).head(20))
print("序列输入 std 最偏离1的特征:")
display(seq_scale_report.sort_values("std_dev_from_1", ascending=False).head(20))

seq_warn = seq_scale_report[(seq_scale_report["seq_abs_max"] > 10) | (seq_scale_report["std_dev_from_1"] > 0.2)]
print("序列输入疑似异常特征数量:", len(seq_warn))
display(seq_warn.head(50))


In [ ]:
# =========================
# Sequence buy/sell signal models: LSTM / CNN / MLP
# =========================

# This stage only produces ML signals: buy_prob and sell_prob.
# T+0 state machine, sizing, and execution backtest are handled later.

import importlib
import json
import model_signals
import ensemble
importlib.reload(model_signals)
importlib.reload(ensemble)
from model_signals import train_dual_transformer_lstm_signals, train_dual_lstm_signals, train_dual_cnn_signals, train_dual_mlp_signals
from ensemble import build_ensemble_signal_result

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEQUENCE_SIGNAL_MODEL_NAMES = MVP_CONFIG.get("sequence_signal_models", ["transformer_lstm", "lstm", "cnn", "mlp"])
sequence_signal_trainers = {
    "transformer_lstm": train_dual_transformer_lstm_signals,
    "lstm": train_dual_lstm_signals,
    "cnn": train_dual_cnn_signals,
    "mlp": train_dual_mlp_signals,
}

sequence_signal_results = {}
for model_key in SEQUENCE_SIGNAL_MODEL_NAMES:
    trainer = sequence_signal_trainers[model_key]
    print(f"\nTraining {model_key.upper()} buy/sell signal models on {DEVICE}...")
    sequence_signal_results[model_key] = trainer(
        x_train_seq=X_train_seq,
        y_train_buy_seq=y_train_seq,
        y_train_sell_seq=y_train_sell_seq,
        x_valid_seq=X_valid_seq,
        y_valid_buy_seq=y_valid_seq,
        y_valid_sell_seq=y_valid_sell_seq,
        x_test_seq=X_test_seq,
        y_test_buy_seq=y_test_seq,
        y_test_sell_seq=y_test_sell_seq,
        valid_df=valid_df,
        test_df=test_df,
        valid_index_seq=valid_index_seq,
        test_index_seq=test_index_seq,
        config=MVP_CONFIG,
        device=DEVICE,
    )
    print(json.dumps(sequence_signal_results[model_key]["summary"], ensure_ascii=False, indent=2))

all_signal_results = {}
if "logit_result" in globals():
    all_signal_results["logistic_regression"] = logit_result
if "rf_signal_result" in globals():
    all_signal_results["random_forest"] = rf_signal_result
all_signal_results.update(sequence_signal_results)

ensemble_signal_result = None
if len(all_signal_results) >= 2:
    ensemble_signal_result = build_ensemble_signal_result(
        all_signal_results,
        config=MVP_CONFIG,
    )
    all_signal_results["auc_weighted_ensemble"] = ensemble_signal_result

primary_sequence_model_name = MVP_CONFIG.get("primary_signal_model", "transformer_lstm")
if primary_sequence_model_name not in sequence_signal_results:
    primary_sequence_model_name = next(iter(sequence_signal_results))
primary_sequence_model_result = sequence_signal_results[primary_sequence_model_name]

if MVP_CONFIG.get("use_ensemble_primary", True) and ensemble_signal_result is not None:
    primary_signal_model_name = "auc_weighted_ensemble"
    primary_signal_result = ensemble_signal_result
else:
    primary_signal_model_name = primary_sequence_model_name
    primary_signal_result = primary_sequence_model_result

# Compatibility variables for later diagnostic/export cells.
model = primary_sequence_model_result["buy_model"]
best_auc = primary_signal_result["summary"].get("valid_buy_auc", np.nan)
test_auc = primary_signal_result["summary"].get("test_buy_auc", np.nan)
summary = primary_signal_result["summary"]

valid_backtest_result = primary_signal_result["valid_signals"].copy()
valid_backtest_result["pred_prob"] = valid_backtest_result["buy_prob"]
valid_backtest_result["y_true"] = valid_backtest_result["buy_label_true"]
valid_backtest_result["pred_label_05"] = (valid_backtest_result["pred_prob"] >= 0.5).astype(int)

backtest_result = primary_signal_result["test_signals"].copy()
backtest_result["pred_prob"] = backtest_result["buy_prob"]
backtest_result["y_true"] = backtest_result["buy_label_true"]
backtest_result["pred_label_05"] = (backtest_result["pred_prob"] >= 0.5).astype(int)

threshold_diagnostics = (
    primary_signal_result["threshold_diagnostics"]
    .loc[lambda x: x["side"] == "buy"]
    .drop(columns=["side"])
    .reset_index(drop=True)
)
group_analysis = (
    primary_signal_result["group_analysis"]
    .loc[lambda x: x["side"] == "buy"]
    .drop(columns=["side"])
    .reset_index(drop=True)
)

model_signal_summaries = pd.DataFrame([
    {
        "model": model_key,
        "valid_buy_auc": result["summary"].get("valid_buy_auc"),
        "test_buy_auc": result["summary"].get("test_buy_auc"),
        "valid_sell_auc": result["summary"].get("valid_sell_auc"),
        "test_sell_auc": result["summary"].get("test_sell_auc"),
        "test_buy_positive_rate": result["summary"].get("test_buy_positive_rate"),
        "test_sell_positive_rate": result["summary"].get("test_sell_positive_rate"),
        "output_dir": str(result["output_dir"]),
    }
    for model_key, result in all_signal_results.items()
])

print("Primary signal model:", primary_signal_model_name)
display(model_signal_summaries)
display(threshold_diagnostics)


In [ ]:
# Migrated to model_signals.py and the sequence signal cell above.
# Current stage focuses on buy_prob / sell_prob signal outputs; old mixed backtest logic is disabled.


In [ ]:
# Migrated to model_signals.py and the sequence signal cell above.
# Current stage focuses on buy_prob / sell_prob signal outputs; old mixed backtest logic is disabled.


In [ ]:
# Migrated to model_signals.py and the sequence signal cell above.
# Current stage focuses on buy_prob / sell_prob signal outputs; old mixed backtest logic is disabled.


In [ ]:
# Migrated to model_signals.py and the sequence signal cell above.
# Current stage focuses on buy_prob / sell_prob signal outputs; old mixed backtest logic is disabled.


In [ ]:
# Migrated to model_signals.py and the sequence signal cell above.
# Current stage focuses on buy_prob / sell_prob signal outputs; old mixed backtest logic is disabled.


In [ ]:
# Migrated to model_signals.py and the sequence signal cell above.
# Current stage focuses on buy_prob / sell_prob signal outputs; old mixed backtest logic is disabled.


In [ ]:
# Migrated to model_signals.py and the sequence signal cell above.
# Current stage focuses on buy_prob / sell_prob signal outputs; old mixed backtest logic is disabled.


In [ ]:
# Migrated to model_signals.py and the sequence signal cell above.
# Current stage focuses on buy_prob / sell_prob signal outputs; old mixed backtest logic is disabled.


In [ ]:
# =========================
# Model signal comparison
# =========================

from pathlib import Path
import json

COMPARE_MODEL_DIR = Path(MVP_CONFIG["diagnostics_dir"])
COMPARE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

comparison_rows = []

if "logit_summary" in globals():
    comparison_rows.append({"model": "logistic_regression", **logit_summary})
else:
    print("Logistic Regression variables missing: run the logit signal cell first")

if "rf_summary" in globals():
    comparison_rows.append({"model": "random_forest", **rf_summary})
else:
    print("RandomForest variables missing: run the RF signal cell first")

if "all_signal_results" in globals():
    for model_key, result in all_signal_results.items():
        comparison_rows.append({"model": model_key, **result["summary"]})
elif "sequence_signal_results" in globals():
    for model_key, result in sequence_signal_results.items():
        comparison_rows.append({"model": model_key, **result["summary"]})
else:
    print("Sequence model variables missing: run the sequence signal cell first")

model_signal_comparison = pd.DataFrame(comparison_rows)
model_signal_comparison.to_csv(COMPARE_MODEL_DIR / "model_signal_comparison.csv", index=False)
with open(COMPARE_MODEL_DIR / "model_signal_comparison.json", "w", encoding="utf-8") as f:
    json.dump(model_signal_comparison.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

metric_cols = [
    "model",
    "valid_buy_auc", "test_buy_auc",
    "valid_sell_auc", "test_sell_auc",
    "valid_buy_f1_05", "test_buy_f1_05",
    "valid_sell_f1_05", "test_sell_f1_05",
]
print("Model signal comparison saved to:", COMPARE_MODEL_DIR / "model_signal_comparison.csv")
display(model_signal_comparison[[c for c in metric_cols if c in model_signal_comparison.columns]])


## 真实交易 pandas 回测

这里不再使用重叠的 `future_return` 直接乘信号，而是模拟固定持有期交易：信号出现后下一根K线开盘买入，持有 `HORIZON` 根K线后收盘卖出，扣买卖双边成本，并且不允许持仓重叠、不隔夜。


In [ ]:
# =========================
# 真实交易框架：事件驱动 pandas 回测
# =========================

from pathlib import Path
import json
from mvp_experiments import run_fixed_horizon_event_backtest

REAL_BT_DIR = Path("outputs/diagnostics/realistic_backtest")
REAL_BT_DIR.mkdir(parents=True, exist_ok=True)

ONE_SIDE_COST = MVP_CONFIG["one_side_cost"]
ROUND_TRIP_COST = MVP_CONFIG["round_trip_cost"]
HOLD_BARS = MVP_CONFIG["horizon"]

# 固定使用验证集 buy 概率 q95 作为交易阈值。
# 这样阈值机制更稳定，不再根据验证集偶然收益选择阈值。
FIXED_BT_QUANTILE = float(MVP_CONFIG.get("fixed_threshold_quantile", 0.95))

threshold_candidates = []
for q in [0.90, 0.925, 0.95, 0.975]:
    threshold = valid_backtest_result["pred_prob"].quantile(q)
    valid_trades_q, valid_equity_q, valid_stats_q = run_fixed_horizon_event_backtest(
        valid_backtest_result,
        threshold=threshold,
        horizon=HOLD_BARS,
        round_trip_cost=ROUND_TRIP_COST,
    )
    valid_stats_q["quantile"] = q
    valid_stats_q["threshold"] = float(threshold)
    threshold_candidates.append(valid_stats_q)

validation_threshold_diagnostics = pd.DataFrame(threshold_candidates)
validation_threshold_diagnostics.to_csv(REAL_BT_DIR / "validation_threshold_diagnostics.csv", index=False)

SELECTED_BT_QUANTILE = FIXED_BT_QUANTILE
SELECTED_BT_THRESHOLD = float(valid_backtest_result["pred_prob"].quantile(SELECTED_BT_QUANTILE))

realistic_trades, realistic_equity, realistic_bt_stats = run_fixed_horizon_event_backtest(
    backtest_result,
    threshold=SELECTED_BT_THRESHOLD,
    horizon=HOLD_BARS,
    round_trip_cost=ROUND_TRIP_COST,
)
realistic_bt_stats["selected_by"] = "fixed_validation_buy_probability_quantile"
realistic_bt_stats["selected_quantile"] = SELECTED_BT_QUANTILE

realistic_trades.to_csv(REAL_BT_DIR / "realistic_trades_selected_threshold.csv", index=False)
pd.Series(realistic_bt_stats).to_csv(REAL_BT_DIR / "realistic_backtest_stats_selected_threshold.csv")

# 测试集分位数阈值扫描仅作诊断，不用于选择最终阈值。
realistic_threshold_rows = []
for q in [0.90, 0.925, 0.95, 0.975]:
    threshold = backtest_result["pred_prob"].quantile(q)
    trades_q, equity_q, stats_q = run_fixed_horizon_event_backtest(
        backtest_result,
        threshold=threshold,
        horizon=HOLD_BARS,
        round_trip_cost=ROUND_TRIP_COST,
    )
    stats_q["quantile"] = q
    stats_q["threshold"] = float(threshold)
    realistic_threshold_rows.append(stats_q)

realistic_threshold_diagnostics = pd.DataFrame(realistic_threshold_rows)
realistic_threshold_diagnostics.to_csv(REAL_BT_DIR / "test_threshold_diagnostics_diagnostic_only.csv", index=False)

realistic_summary = {
    "backtest_type": "fixed_horizon_event_backtest_buy_side_from_dual_signal_model",
    "entry_rule": "signal at bar t when buy_prob >= fixed validation q95, enter next bar open",
    "exit_rule": "exit after horizon bars at close",
    "no_overnight": True,
    "no_overlap": True,
    "label_mode": MVP_CONFIG.get("label_mode"),
    "label_threshold": float(LABEL_THRESHOLD),
    "one_side_cost": float(ONE_SIDE_COST),
    "round_trip_cost": float(ROUND_TRIP_COST),
    "selected_threshold": float(SELECTED_BT_THRESHOLD),
    "selected_quantile": float(SELECTED_BT_QUANTILE),
    "selected_threshold_test_stats": realistic_bt_stats,
    "output_files": {
        "selected_threshold_trades": str(REAL_BT_DIR / "realistic_trades_selected_threshold.csv"),
        "selected_threshold_stats": str(REAL_BT_DIR / "realistic_backtest_stats_selected_threshold.csv"),
        "validation_threshold_diagnostics": str(REAL_BT_DIR / "validation_threshold_diagnostics.csv"),
        "test_threshold_diagnostics_diagnostic_only": str(REAL_BT_DIR / "test_threshold_diagnostics_diagnostic_only.csv"),
    },
}

with open(REAL_BT_DIR / "realistic_backtest_summary.json", "w", encoding="utf-8") as f:
    json.dump(realistic_summary, f, ensure_ascii=False, indent=2)

print("真实交易回测结果已保存到:", REAL_BT_DIR)
print(pd.Series(realistic_bt_stats))
print("验证集固定阈值诊断:")
display(validation_threshold_diagnostics)
print("测试集阈值诊断，仅供观察:")
display(realistic_threshold_diagnostics)
display(realistic_trades.head())

if len(realistic_equity):
    plt.figure(figsize=(12, 5))
    plt.plot(realistic_equity.index, realistic_equity.values, label="Realistic Event Backtest")
    plt.title("Realistic Fixed-Horizon Backtest Equity Curve")
    plt.grid(True)
    plt.legend()
    plt.show()
else:
    print("没有触发交易。")


In [ ]:
# =========================
# Optional explainability and dual-signal T+0 research backtest
# =========================

# This cell is intentionally downstream of model_signal_comparison.
# It leaves auditable files for feature importance and a first realistic state-machine check.

from pathlib import Path
from explainability import (
    compute_permutation_importance_binary,
    compute_torch_gradient_importance,
    try_compute_shap_summary,
    export_model_explainability,
)
from risk_management import run_t0_dual_signal_backtest, export_t0_backtest_result

RESEARCH_DIR = Path(MVP_CONFIG["diagnostics_dir"]) / "research_checks"
EXPLAIN_DIR = RESEARCH_DIR / "explainability"
T0_DIR = RESEARCH_DIR / "dual_signal_t0_backtest"
RESEARCH_DIR.mkdir(parents=True, exist_ok=True)

permutation_results = {}
shap_results = {}
if "rf_buy_model" in globals() and "rf_sell_model" in globals():
    permutation_results["rf_buy"] = compute_permutation_importance_binary(
        rf_buy_model,
        test_scaled.loc[test_index_seq],
        y_test_seq,
        feature_cols,
        n_repeats=MVP_CONFIG.get("permutation_repeats", 3),
        random_state=MVP_CONFIG["random_state"],
    ).head(MVP_CONFIG.get("explainability_top_features", 30))
    permutation_results["rf_sell"] = compute_permutation_importance_binary(
        rf_sell_model,
        test_scaled.loc[test_index_seq],
        y_test_sell_seq,
        feature_cols,
        n_repeats=MVP_CONFIG.get("permutation_repeats", 3),
        random_state=MVP_CONFIG["random_state"],
    ).head(MVP_CONFIG.get("explainability_top_features", 30))
    shap_results["rf_buy"] = try_compute_shap_summary(rf_buy_model, test_scaled.loc[test_index_seq], feature_cols)

# Gradient importance is model-specific and runs on the primary sequence model.
gradient_results = {}
if "primary_sequence_model_result" in globals():
    gradient_results["primary_buy"] = compute_torch_gradient_importance(
        primary_sequence_model_result["buy_model"],
        X_test_seq,
        feature_cols,
        device=DEVICE,
        max_samples=1024,
    ).head(MVP_CONFIG.get("explainability_top_features", 30))
    gradient_results["primary_sell"] = compute_torch_gradient_importance(
        primary_sequence_model_result["sell_model"],
        X_test_seq,
        feature_cols,
        device=DEVICE,
        max_samples=1024,
    ).head(MVP_CONFIG.get("explainability_top_features", 30))

explainability_manifest = export_model_explainability(
    EXPLAIN_DIR,
    summary={
        "primary_signal_model": globals().get("primary_signal_model_name"),
        "primary_sequence_model": globals().get("primary_sequence_model_name"),
        "feature_count": len(feature_cols),
        "permutation_repeats": MVP_CONFIG.get("permutation_repeats", 3),
        "note": "Permutation AUC drop and input-gradient importance are research diagnostics, not trading rules.",
    },
    permutation_results=permutation_results,
    gradient_results=gradient_results,
    shap_results=shap_results,
)

if "primary_signal_result" in globals():
    t0_signal_df = primary_signal_result["test_signals"].copy()
    buy_threshold = float(primary_signal_result["valid_signals"]["buy_prob"].quantile(MVP_CONFIG.get("fixed_threshold_quantile", 0.95)))
    sell_threshold = float(primary_signal_result["valid_signals"]["sell_prob"].quantile(MVP_CONFIG.get("fixed_threshold_quantile", 0.95)))
    t0_trades, t0_equity, t0_stats = run_t0_dual_signal_backtest(
        t0_signal_df,
        buy_threshold=buy_threshold,
        sell_threshold=sell_threshold,
        config=MVP_CONFIG,
    )
    export_t0_backtest_result(T0_DIR, t0_trades, t0_equity, t0_stats)
    print("T+0 dual-signal research backtest saved to:", T0_DIR)
    display(pd.Series(t0_stats))
    display(t0_trades.head())

print("Explainability files saved to:", EXPLAIN_DIR)
print(explainability_manifest)


In [ ]:
# =========================
# Walk-forward research runner
# =========================

# Default is off because a full walk-forward run can be slow.
# Set MVP_CONFIG["run_walk_forward_in_notebook"] = True or call run_walk_forward_signal_research(...) manually.

import importlib
import walk_forward_runner
importlib.reload(walk_forward_runner)
from walk_forward_runner import run_walk_forward_signal_research

if MVP_CONFIG.get("run_walk_forward_in_notebook", False):
    walk_forward_result = run_walk_forward_signal_research(
        data=df_model,
        feature_cols=feature_cols,
        config=MVP_CONFIG,
        output_dir=Path(MVP_CONFIG["diagnostics_dir"]) / "walk_forward",
        model_names=MVP_CONFIG.get("walk_forward_model_names", ["logistic_regression", "random_forest"]),
        device=DEVICE if "DEVICE" in globals() else None,
    )
    print("Walk-forward output dir:", walk_forward_result["output_dir"])
    display(walk_forward_result["fold_summary"])
    display(pd.Series(walk_forward_result["quality_report"]))
else:
    print("Walk-forward runner is ready. Set run_walk_forward_in_notebook=True to execute.")


In [ ]:
# # =========================
# # 鲁棒性检测：固定 LOOKBACK=6, HORIZON=12，多随机种子重复训练
# # =========================

# import importlib
# import mvp_experiments
# importlib.reload(mvp_experiments)

# from mvp_experiments import run_seed_robustness

# ROBUSTNESS_LOOKBACK = 6
# ROBUSTNESS_HORIZON = 12
# ROBUSTNESS_SEEDS = [7, 11, 19, 23, 42]

# robustness_runs, robustness_summary = run_seed_robustness(
#     base_df=df.copy().sort_index(),
#     feature_cols=feature_cols,
#     target_col=target_col,
#     config=MVP_CONFIG,
#     horizon=ROBUSTNESS_HORIZON,
#     lookback=ROBUSTNESS_LOOKBACK,
#     seeds=ROBUSTNESS_SEEDS,
#     output_dir=f"outputs/diagnostics/robustness/lb{ROBUSTNESS_LOOKBACK}_h{ROBUSTNESS_HORIZON}",
#     device=device,
#     verbose=True,
# )

# print("鲁棒性检测明细已保存到:", f"outputs/diagnostics/robustness/lb{ROBUSTNESS_LOOKBACK}_h{ROBUSTNESS_HORIZON}")
# display(robustness_runs)
# display(robustness_summary)


## vn.py 回测

下面的 vn.py 回测保留为平台接入示例。注意：当前 vn.py 策略是概率状态切换，不等同于上面的固定持有期真实交易框架；正式比较策略绩效时优先看 pandas 事件驱动回测。


In [ ]:
# vn.py 相关依赖检查。
# 如果这里提示缺包，再单独手动运行：%pip install vnpy_ctastrategy

from vnpy_backtest import check_vnpy_available

VNPY_AVAILABLE, VNPY_INFO = check_vnpy_available()
print("vn.py available:", VNPY_AVAILABLE)
if not VNPY_AVAILABLE:
    print({k: v for k, v in VNPY_INFO.items() if k.endswith("error")})


In [ ]:
# vn.py 合约参数

SYMBOL = MVP_CONFIG["symbol"]
if VNPY_INFO.get("core"):
    EXCHANGE = VNPY_INFO["Exchange"].SSE
    VT_SYMBOL = f"{SYMBOL}.{EXCHANGE.value}"
    print(VT_SYMBOL)
else:
    EXCHANGE = None
    VT_SYMBOL = None


In [ ]:
# 1) 将 OHLCV 数据写入 vn.py 数据库。
# vn.py 没有单独的 5分钟 Interval 枚举，所以这里仍然用 Interval.MINUTE 表示分钟级K线。

from vnpy_backtest import save_df_to_vnpy_database

saved_count = save_df_to_vnpy_database(
    df,
    symbol=SYMBOL,
    exchange=EXCHANGE,
    gateway_name="LOCAL_5MIN",
    vnpy_info=VNPY_INFO,
)
print("saved bars:", saved_count)


In [ ]:
# 2) 导出模型预测信号。

from vnpy_backtest import export_signal_file

signal_path, signal_df = export_signal_file(
    backtest_result,
    MVP_CONFIG["vnpy_signal_path"],
    prob_col="pred_prob",
)
print(signal_path)
display(signal_df.head())


In [ ]:
# 3) vn.py CTA 策略定义已迁移到 vnpy_backtest.build_ml_signal_long_only_strategy。
# 当前策略仍是平台接入示例，不等同于最终 T+0 高抛低吸双向动作状态机。


In [ ]:
# 4) 运行 vn.py 回测。
# rate=0.001 表示 0.1% 佣金；slippage=0.05 表示每股 0.05 元滑点。

from vnpy_backtest import run_vnpy_backtest

if VNPY_AVAILABLE:
    VNPY_BT_THRESHOLD = SELECTED_BT_THRESHOLD if "SELECTED_BT_THRESHOLD" in globals() else backtest_result["pred_prob"].quantile(MVP_CONFIG.get("fixed_threshold_quantile", 0.95))
    engine, daily_result, stats = run_vnpy_backtest(
        vt_symbol=VT_SYMBOL,
        signal_path=signal_path,
        backtest_result=backtest_result,
        threshold=VNPY_BT_THRESHOLD,
        vnpy_info=VNPY_INFO,
        fixed_size=100,
        capital=1_000_000,
    )
    display(pd.Series(stats))
else:
    engine = None
    daily_result = pd.DataFrame()
    stats = {}
    print("vn.py unavailable, backtest skipped")


In [ ]:
# 5) 查看 vn.py 回测成交、逐日结果和资金曲线。
if engine is not None:
    trades = engine.get_all_trades()
    orders = engine.get_all_orders()

    print("trades:", len(trades))
    print("orders:", len(orders))

    if daily_result is not None and not daily_result.empty:
        display(daily_result.tail())
    else:
        print("daily_result is empty")
else:
    print("vn.py engine is None, skipped")


In [ ]:
# 6) 画 vn.py 回测资金曲线和回撤。
import matplotlib.pyplot as plt

if daily_result is not None and not daily_result.empty and "balance" in daily_result.columns:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

    axes[0].plot(daily_result.index, daily_result["balance"], label="vn.py Strategy Balance")
    axes[0].set_title("vn.py MVP Backtest Balance")
    axes[0].grid(True)
    axes[0].legend()

    if "drawdown" in daily_result.columns:
        axes[1].plot(daily_result.index, daily_result["drawdown"], color="red", label="Drawdown")
        axes[1].set_title("vn.py Drawdown")
        axes[1].grid(True)
        axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print("daily_result is empty or has no balance column")
